# CRIVO — fase 7b: o 300M aprende a conversar (receita v7c, A100)

**De onde parte:** da base de **290M** da fase 7a (`CRIVO/transformer-crivo-300m/resumo_fase7a.json`, conferida pelo SHA):
o v7c (46M) crescido em largura e profundidade e depois lendo 2,95 bilhões de tokens (perda no texto próprio 2,804 → 2,607;
na web 2,783 → 2,541; cópia do contexto 59,9% → 81,2%). Ela lê melhor, mas ainda não foi ensinada a conversar.

**O que faz:** a mesma receita do v7c (os mesmos dados, `gerar_v7c.py`, a mesma substituição por situação, a mesma guarda no
dev e a mesma média móvel dos pesos), com três mudanças, fixadas antes de rodar:
1. **a base** é a da 7a (290M, 40 camadas, 768 de largura) no lugar da 6a (46M);
2. **a régua compara com o v7c**, o modelo que está no chat hoje (sha `e4067e3b…`), e não mais com o v4: a guarda no dev e a
   linha de base pareada dos blinds usam o v7c. O arquivo da linha de base mantém o nome `linhas_v4_base.json` (é o nome que
   `comparar_v6.py` lê), mas leva os pesos do v7c, com o SHA dentro; nas tabelas, a coluna "v4" é o chat atual (v7c);
3. **o tamanho**: taxa de aprendizado 6e-5 → 6e-6 (o 46M usou 1e-4; modelo maior, passo menor), o lote em pedaços conforme a
   memória da GPU e checkpoint a cada 1.000 passos (cada um tem uns 6 GB).

Os critérios de aprovação **não mudam** (`comparar_v6.py`): blind v3 com IC > 0 **contra o v7c**, outros blinds ≥ −1,5, frases
reais ≥ 12/15, usa o fato ≥ 0,85 e urgência ≥ a do chat, **nas duas rodadas** (sementes 20261019 e 20261020). Passar não liga
nada: os pesos só vão para o chat por decisão do criador, depois da conversão para NumPy e do motor do celular.

**Dados do professor:** as falas da pessoa do professor (modelo de terceiros, não humanas) entram como no v7c; o resumo registra
a fração das exposições de conversa que veio delas (`professor_fracao_exposicoes`).

> **GPU → A100 → Executar tudo** (rodada 1). Depois, de novo, para a rodada 2 (mesma GPU). Na terceira vez, só compara.
> Pode parar e continuar: "Executar tudo" de novo continua do checkpoint. Saída: `CRIVO/transformer-crivo-300m-conversa-v7c/`.
> Os checkpoints substituídos vão para a lixeira do Drive: esvazie a lixeira se o espaço apertar.


In [ ]:
# Célula 0 — GPU, Drive e a base crescida da fase 6a
import os, sys, json, shutil, subprocess, hashlib, time, math, random, glob, re, gzip
from pathlib import Path
from datetime import datetime
from collections import Counter
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100, L4 ou T4.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # T4 e mais antigas: float16 (bfloat16 lá é emulado, lento e sem atenção eficiente)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-300m" / "resumo_fase7a.json").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/transformer-crivo-300m/resumo_fase7a.json. Rode na conta onde a fase 7a treinou.")
R7A = json.loads((CRIVO / "transformer-crivo-300m" / "resumo_fase7a.json").read_text())
if not R7A.get("terminou") or not R7A.get("pesos_sha256"):
    raise SystemExit("ERRO: a fase 7a ainda não terminou.")
BASE = Path(R7A["pasta"]) / "pesos.pt"
if not BASE.is_file():   # o Drive pode estar montado em outro caminho
    BASE = CRIVO / "transformer-crivo-300m" / Path(R7A["pasta"]).name / "pesos.pt"
if sha256(BASE) != R7A["pesos_sha256"]:
    raise SystemExit(f"ERRO: {BASE} não tem o SHA do resumo da 7a ({R7A['pesos_sha256'][:8]}…).")
BASE_SHA = R7A["pesos_sha256"]
PARTIDA = "7a"   # receita longa do v7c, a partir da base de 290M da 7a
USAR_PROFESSOR = True   # False: só as falas escritas pelo assistente
# O chat de hoje: o 46M v7c (rodada 2). É a régua: guarda no dev e linha de base pareada.
CHAT_SHA = "e4067e3b4c9705fde0a2063cc7a0da958ebad196ca193a672f75a1a10cbfda0a"
CHAT_PASTA = CRIVO / "transformer-crivo-46m-conversa-v7c"
CHAT_PESOS = next((p for p in sorted(CHAT_PASTA.glob("rodada*/conversa_*/pesos.pt")) if sha256(p) == CHAT_SHA), None)
if CHAT_PESOS is None:
    raise SystemExit("ERRO: não achei os pesos do v7c do chat (sha e4067e3b…) em CRIVO/transformer-crivo-46m-conversa-v7c.")
SAIDA = CRIVO / "transformer-crivo-300m-conversa-v7c"
SAIDA.mkdir(parents=True, exist_ok=True)
DADOS_DRIVE = CRIVO / "dados-fase6b"
DADOS_DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/fase6b")
LOCAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
print("CRIVO:", CRIVO, "| base 7a:", BASE, BASE_SHA[:12], "| chat v7c:", CHAT_PESOS, flush=True)


In [ ]:
# Célula 1 — código do CRIVO
REPO = Path("/content/CRIVO")
BRANCH = "claude/happy-mendel-wijsxp"   # depois do merge: "main"
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
for p in (str(REPO), str(REPO / "scripts"), str(REPO / "experimentos" / "fase6b_conversa_humana"),
          str(REPO / "experimentos" / "fase5_conversa_20261013"), str(REPO / "experimentos" / "v6_entendimento_20261019")):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase6, gerar_fase6b, extras_fase6b, cenas_v4
import baixar_fontes_linguagem
import importlib.util
_spec = importlib.util.spec_from_file_location("formato_fase5", REPO / "artefatos" / "conversa_v4f_pt" / "formato_fase5.py")
formato_fase5 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(formato_fase5)
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(BASE.parent / "tokenizer.json"))
ESP = {t: TOK.token_to_id(t) for t in ("<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>")}
DOC, FIM = ESP["<documento>"], ESP["<fim>"]
QUEM = gerar_fase6b.resumo_identidade()
# Fase 5: dados v1–v3 do repositório (SHA conferido) e v4 gerada aqui, como no notebook da v4.
EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
M = {n: json.loads((EXP5 / f).read_text(encoding="utf-8")) for n, f in (("v1", "manifesto.json"), ("v2", "manifesto_v2.json"),
                                                                       ("v3", "manifesto_v3.json"), ("v4", "manifesto_v4.json"))}
for arq, esperado in (("dados_fase5.json.gz", M["v1"]["arquivo_gz_sha256"]), ("dados_fase5_v2.json.gz", M["v2"]["arquivo_gz_sha256"]),
                      ("dados_fase5_v3.json.gz", M["v3"]["arquivo_gz_sha256"]), ("sentido.py", M["v4"]["sentido_sha256"]),
                      ("cenas_v4.py", M["v4"]["cenas_v4_sha256"]), ("gerar_v4.py", M["v4"]["gerar_v4_sha256"])):
    if sha256(EXP5 / arq) != esperado:
        raise SystemExit(f"ERRO: {arq} difere do manifesto")
V4_SHA_REGISTRADO = M["v4"]["dados_sha256"]
subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
V4_SHA = json.loads((EXP5 / "manifesto_v4.json").read_text(encoding="utf-8"))["dados_sha256"]
print("dados da fase 5 v4:", "conferidos" if V4_SHA == V4_SHA_REGISTRADO else f"AVISO: sha {V4_SHA[:12]} ≠ registrado {V4_SHA_REGISTRADO[:12]}", flush=True)
import sentido
sentido_ok = cenas_v4.registrar_no_sentido(sentido)
ACEITAS_IDENT = extras_fase6b.aceitas_na_fase5(sentido.norm)   # respostas do IDENTIDADE.md, certas nessas situações
F5 = {n: json.loads(gzip.open(EXP5 / f).read().decode("utf-8")) for n, f in (("v1", "dados_fase5.json.gz"), ("v2", "dados_fase5_v2.json.gz"),
                                                                             ("v3", "dados_fase5_v3.json.gz"), ("v4", "dados_fase5_v4.json.gz"))}
F_SENTIDO = {k: v["sentido"]["certo"] for k, v in json.loads((EXP5 / "resultado_v4_resumo.json").read_text())["variantes"]["F"]["testes"].items()
             if isinstance(v, dict) and "sentido" in v}
# Dados v6 (falas da pessoa escritas pelo assistente, com autorização do criador; não são humanas)
EXP6 = REPO / "experimentos" / "v6_entendimento_20261019"
import comparar_v6
import substituicao_v6c, formas_grok
# Fase 7b: o "chat" da régua é o v7c (R4/P4 mantêm os nomes do notebook do v7c).
R4 = {"pesos_sha256": CHAT_SHA, "pasta": str(CHAT_PESOS.parent)}
P4 = CHAT_PESOS
print("partida:", PARTIDA, "| pesos do chat (v7c):", P4, flush=True)


def com_o_chat(f):
    """Roda f() com o 46M v7c no lugar do modelo (as funções de fala leem o nome global `modelo`)."""
    global modelo
    proprio = modelo
    salvo = torch.load(P4, map_location="cpu", weights_only=True)
    chat = lp.LinguagemProfunda(lp.Configuracao(**dict(salvo["config"], dropout=0.0))).to(dispositivo)
    chat.load_state_dict(salvo["modelo"]); del salvo
    modelo = chat
    try:
        return f()
    finally:
        modelo = proprio
        del chat
        torch.cuda.empty_cache()

# Rodada: a primeira sem resumo. Com as duas prontas, só compara.
SEMENTES = {1: 20261019, 2: 20261020}
RODADA = next((n for n in SEMENTES if not (SAIDA / f"rodada{n}" / "resumo_v6.json").is_file()), None)
if RODADA is None:
    comparar_v6.principal(SAIDA, com_urgencia=True)
    raise SystemExit("As duas rodadas já terminaram; acima, a comparação. Avise o assistente.")
SEED = SEMENTES[RODADA]
SAIDA_R = SAIDA / f"rodada{RODADA}"
SAIDA_R.mkdir(parents=True, exist_ok=True)
print(f"RODADA {RODADA} (semente {SEED}) → {SAIDA_R}", flush=True)
print("commit:", COMMIT, "| quem:", QUEM, "| v4 F (chat) no sentido:", F_SENTIDO, flush=True)


In [ ]:
# Célula 1b — falas da pessoa: as do assistente e as do professor (feitas por professor_v7_colab.ipynb), e os dados v7c
import professor_v7 as PV, detector_gatilhos as DG, gerar_v6 as G6, gerar_v6b as G6B, formas_v7
M7_REPO = json.loads((EXP6 / "manifesto_v7c.json").read_text(encoding="utf-8"))   # o manifesto sem professor, versionado
for arq in ("formas_v7", "formas_v7_a", "formas_v7_b", "formas_v7_c", "formas_v7_d", "formas_v7_e", "formas_v7_saude",
            "formas_v7_identidade", "formas_v7c", "coerencia_professor", "detector_gatilhos", "substituicao_v6c"):
    if sha256(EXP6 / f"{arq}.py") != M7_REPO[f"{arq}_sha256"]:
        raise SystemExit(f"ERRO: {arq}.py difere do manifesto_v7c.json")
PROF_DIR = CRIVO / "dados-professor-v7"
PROF_DIR.mkdir(parents=True, exist_ok=True)
PROF_ARQ = PROF_DIR / "falas_professor_v7.jsonl.gz"
if USAR_PROFESSOR and not PROF_ARQ.is_file():
    raise SystemExit("Falta o arquivo do professor. Rode antes notebooks/professor_v7_colab.ipynb (numa sessão própria), depois "
                     "Ambiente de execução → Desconectar e excluir ambiente, e rode este de novo. Para treinar sem o professor: "
                     "USAR_PROFESSOR = False na célula 0.")
if USAR_PROFESSOR:
    MP = json.loads((PROF_DIR / "manifesto_professor_v7.json").read_text())
    print(f"professor: {MP['professor']['modelo']} ({MP['professor']['licenca']}) | falas lidas {MP['falas_lidas']} | "
          f"aprovadas na conferência {MP['aprovadas_na_conferencia']} | fora na leitura {MP['fora_na_leitura']}", flush=True)
    rr = random.Random(3)
    with gzip.open(PROF_ARQ, "rt", encoding="utf-8") as f:
        _amostra = [json.loads(l) for l in f]
    for x in rr.sample(_amostra, min(30, len(_amostra))):
        print(f"   [{x['situacao']}] {x['fala']}", flush=True)
    del _amostra

# os dados v7c (determinísticos dado o arquivo do professor)
subprocess.run([sys.executable, str(EXP6 / "gerar_v7c.py")] + (["--professor", str(PROF_ARQ)] if USAR_PROFESSOR else []),
               cwd=EXP6, check=True)
M6 = json.loads((EXP6 / "manifesto_v7c.json").read_text(encoding="utf-8"))
V6 = json.loads(gzip.open(EXP6 / "dados_v7c.json.gz").read().decode("utf-8"))
print("dados v7c:", M6["tamanhos"], "| falas por origem:", M6["falas_por_origem"], "| filtros:", M6["relatorio"], flush=True)


In [ ]:
# Célula 2 — dados de conversa (uma vez; depois vêm do Drive) e tokens
DADOS = DADOS_DRIVE / "dados_fase6b.json.gz"
if not DADOS.is_file():
    OASST = LOCAL / "oasst2.messages.jsonl.gz"
    if not OASST.is_file():
        baixar_fontes_linguagem.baixar("oasst2", LOCAL)
    LEGENDAS = gerar_fase6b.LEGENDAS_URL
    subprocess.run([sys.executable, str(REPO / "experimentos/fase6b_conversa_humana/gerar_fase6b.py"), "--oasst", str(OASST),
                    "--legendas", LEGENDAS, "--saida", str(LOCAL / "dados")], cwd=REPO, check=True)
    shutil.copyfile(LOCAL / "dados" / "manifesto_fase6b.json", DADOS_DRIVE / "manifesto_fase6b.json")
    shutil.copyfile(LOCAL / "dados" / "dados_fase6b.json.gz", DADOS)
MANIFESTO_DADOS = json.loads((DADOS_DRIVE / "manifesto_fase6b.json").read_text())
with gzip.open(DADOS, "rb") as f:
    bruto = f.read()
assert hashlib.sha256(bruto).hexdigest() == MANIFESTO_DADOS["dados_sha256"], "dados_fase6b.json.gz corrompido"
DADOS_CONVERSA = json.loads(bruto); del bruto
REL_EXTRAS = extras_fase6b.extras(DADOS_CONVERSA, QUEM)   # identidade, lembranças e filtro do OASST2 (determinístico)

def de_fase5(c):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    return {"origem": "fase5", "ctx": {"quem": QUEM, **(c["contexto"] or {})},
            "turnos": t + [{"papel": "usuario", "texto": c["mensagem"]}, {"papel": "assistente", "texto": c["resposta"]}]}
REP_F5, REP_V6 = (1, 3) if PARTIDA == "v4" else (3, 3)
FASE5_TREINO = [c for c in F5["v4"]["treino"] if c["situacao"][5:] not in extras_fase6b.FASE5_SUBSTITUIDAS]
FASE5_FICA, V6_ENTRA, REL_SUBST = substituicao_v6c.substituir(FASE5_TREINO, V6["treino"], REP_F5, REP_V6, 20261020, formas_grok.chave,
                                                                sem_antigos_entram=True)   # identidade: não tem casos na fase 5
substituicao_v6c.conferir(REL_SUBST, REP_V6)   # o peso de cada situação não muda
# v7c: a identidade também não ganha peso. Ela não tem casos na fase 5 (a 6b treina pelo IDENTIDADE.md), então cada caso
# novo de identidade (×REP_V6) tira a mesma conta de exemplos do IDENTIDADE.md (×REP_IDENT) do grupo que cobre a situação.
REP_IDENT = 1 if PARTIDA == "v4" else 4
GRUPOS_IDENT = {"quem_e": ("quem",), "nome_crivo": ("quem",), "ser_humano": ("pessoa", "humano"), "robo": ("robo",), "sente": ("sentir",)}
_novos_ident = Counter(c["situacao"][5:] for c in V6_ENTRA if c["situacao"][5:] in GRUPOS_IDENT)
_rr_ident, _tirar_ident, REL_IDENT = random.Random(20261021), set(), {}
for _s, _n in sorted(_novos_ident.items()):
    _cands = [i for i, e in enumerate(DADOS_CONVERSA["treino"]) if e["origem"] == "identidade"
              and e["chave"].split(":")[1] in GRUPOS_IDENT[_s] and i not in _tirar_ident]
    _k = min(len(_cands) // 2, math.ceil(_n * REP_V6 / REP_IDENT))
    _tirar_ident.update(_rr_ident.sample(_cands, _k))
    REL_IDENT[_s] = {"casos_novos": _n, "exposicoes_novas": _n * REP_V6, "identidade_tirados": _k, "exposicoes_tiradas": _k * REP_IDENT}
DADOS_CONVERSA["treino"] = [e for i, e in enumerate(DADOS_CONVERSA["treino"]) if i not in _tirar_ident]
assert all(abs(r["exposicoes_novas"] - r["exposicoes_tiradas"]) < REP_IDENT for r in REL_IDENT.values()), REL_IDENT
print("identidade sem ganhar peso:", REL_IDENT, flush=True)
print(f"substituição: {len(V6_ENTRA)} casos novos (×{REP_V6}) no lugar de {len(FASE5_TREINO) - len(FASE5_FICA)} antigos (×{REP_F5});"
      f" situações limitadas: { {s: (r['novos'], r['novos_pedidos']) for s, r in REL_SUBST.items() if r['novos'] < r['novos_pedidos']} }", flush=True)
DADOS_CONVERSA["treino"] += [de_fase5(c) for c in FASE5_FICA]
DADOS_CONVERSA["val"] += [de_fase5(c) for c in F5["v4"]["dev"]]
def de_v6(c):
    e = de_fase5(c); e["origem"] = "v6"; return e
DADOS_CONVERSA["treino"] += [de_v6(c) for c in V6_ENTRA]
DADOS_CONVERSA["val"] += [de_v6(c) for c in V6["dev"]]
REL_EXTRAS["v6_treino"], REL_EXTRAS["v6_dev"] = len(V6_ENTRA), len(V6["dev"])
REL_EXTRAS["fase5_substituidos"] = len(FASE5_TREINO) - len(FASE5_FICA)
REL_EXTRAS["v7_casos_por_origem"] = dict(Counter(c.get("origem") for c in V6_ENTRA))
REL_EXTRAS["identidade_sem_ganhar_peso"] = REL_IDENT
REL_EXTRAS["fase5_treino"], REL_EXTRAS["fase5_dev"] = len(F5["v4"]["treino"]), len(F5["v4"]["dev"])
print("acrescentados:", REL_EXTRAS, flush=True)
print("dados:", json.dumps(MANIFESTO_DADOS["por_origem"], ensure_ascii=False), flush=True)
print("relatório:", json.dumps(MANIFESTO_DADOS["relatorio"], ensure_ascii=False), flush=True)

# tokens: cada texto distinto é codificado uma vez, em lote
_textos = set()
for v in DADOS_CONVERSA.values():
    for e in v:
        _textos.add(formato_fase6.limpar(formato_fase6.serializar_contexto(e.get("ctx"))))
        _textos.update(formato_fase6.limpar(t["texto"]) for t in e["turnos"])
_textos = sorted(_textos)
CACHE = dict(zip(_textos, (e.ids for e in TOK.encode_batch(_textos, add_special_tokens=False))))
del _textos
def codificar(texto):
    ids = CACHE.get(texto)
    return ids if ids is not None else TOK.encode(texto, add_special_tokens=False).ids

CTX = 1024
EX = {}   # (parte, origem) -> lista de (ids uint16, alvo bool)
nao_cabe = Counter()
for parte, v in DADOS_CONVERSA.items():
    for e in v:
        try:
            ids, alvo = formato_fase6.montar(codificar, ESP, e.get("ctx"), e["turnos"], contexto=CTX)
        except ValueError:
            nao_cabe[e["origem"]] += 1
            continue
        EX.setdefault((parte, e["origem"]), []).append((np.asarray(ids, np.uint16), np.asarray(alvo, bool)))
TOKENS_ALVO = {f"{p}/{o}": int(sum(a.sum() for _, a in l)) for (p, o), l in sorted(EX.items())}
print("exemplos:", {f"{p}/{o}": len(l) for (p, o), l in sorted(EX.items())}, "| não couberam:", dict(nao_cabe), flush=True)
print("tokens com perda:", TOKENS_ALVO, flush=True)

# leitura: corpus próprio, uma parte do FineWeb-2 (da 6a) e o IDENTIDADE.md
for n in ("treino.bin", "validacao.bin"):
    if not (LOCAL / n).is_file():
        shutil.copyfile(CRIVO / "dados-17m" / n, LOCAL / n)
PROPRIO = np.memmap(LOCAL / "treino.bin", dtype=np.uint16, mode="r")
PROPRIO_VAL = np.memmap(LOCAL / "validacao.bin", dtype=np.uint16, mode="r")
MAN_WEB = json.loads((CRIVO / "corpus-fineweb2-pt" / "manifesto.json").read_text())
for p in (MAN_WEB["partes"][-1], MAN_WEB["val"]):   # a última parte: a que a 6a leu por último
    if not (LOCAL / p["arquivo"]).is_file() or (LOCAL / p["arquivo"]).stat().st_size != 2 * p["tokens"]:
        shutil.copyfile(CRIVO / "corpus-fineweb2-pt" / p["arquivo"], LOCAL / p["arquivo"])
WEB = np.memmap(LOCAL / MAN_WEB["partes"][-1]["arquivo"], dtype=np.uint16, mode="r")
WEB_VAL = np.memmap(LOCAL / MAN_WEB["val"]["arquivo"], dtype=np.uint16, mode="r")
IDENT = np.asarray([DOC] + TOK.encode(formato_fase6.limpar((REPO / "IDENTIDADE.md").read_text(encoding="utf-8")),
                                      add_special_tokens=False).ids + [FIM], np.uint16)
print(f"leitura: próprio {len(PROPRIO)/1e9:.2f} bi | web {len(WEB)/1e6:.0f}M | IDENTIDADE.md {len(IDENT)} tokens", flush=True)


In [ ]:
# Célula 3 — o plano de treino (determinístico: recomeçar dá os mesmos lotes)
if PARTIDA == "v4":   # ajuste leve: uma volta, as falas novas ×3 no lugar das antigas
    REPETIR = {"v6": REP_V6, "fase5": REP_F5, "oasst2": 1, "acervo": 1, "legendas": 1, "identidade": REP_IDENT, "lembranca": 1}
    MAX_PASSOS = 1300
else:                # receita longa da 6b v4, com substituição
    REPETIR = {"v6": REP_V6, "fase5": REP_F5, "oasst2": 4, "acervo": 2, "legendas": 1, "identidade": REP_IDENT, "lembranca": 1}
    MAX_PASSOS = 3400
LOTE, CONVERSA_POR_LOTE = 64, 48
L = CTX + 1

def empacotar(exemplos, rng):
    """Junta conversas em sequências de CTX+1 tokens; o resto da sequência é <fim> sem perda."""
    ordem = rng.permutation(len(exemplos))
    seqs, alvos, ids_cur, alvo_cur = [], [], [], []
    def fechar():
        ids = np.concatenate(ids_cur); alvo = np.concatenate(alvo_cur)
        s = np.full(L, FIM, np.uint16); a = np.zeros(L, bool)
        s[:len(ids)] = ids; a[:len(alvo)] = alvo
        seqs.append(s); alvos.append(a)
    n = 0
    for i in ordem:
        ids, alvo = exemplos[i]
        if n + len(ids) > L and ids_cur:
            fechar(); ids_cur, alvo_cur, n = [], [], 0
        ids_cur.append(ids); alvo_cur.append(alvo); n += len(ids)
    if ids_cur:
        fechar()
    return seqs, alvos

def amostra_por_tokens(lista, tokens, rng, inicio=0):
    """Pega exemplos de `lista` (já embaralhada) a partir de `inicio` até somar `tokens`."""
    out, soma, i = [], 0, inicio
    while soma < tokens and i < len(lista):
        out.append(lista[i]); soma += len(lista[i][0]); i += 1
    return out, i

rng_plano = np.random.default_rng(SEED)
tokens_de = lambda lista: sum(len(i) for i, _ in lista)
por_origem = {o: EX.get(("treino", o), []) for o in REPETIR}
legendas = por_origem["legendas"]
legendas = [legendas[i] for i in rng_plano.permutation(len(legendas))]
resto = sum(REPETIR[o] * tokens_de(l) for o, l in por_origem.items() if o != "legendas")
cabe = MAX_PASSOS * CONVERSA_POR_LOTE * L * 0.92 - resto     # ~92% de cada sequência é ocupada
if tokens_de(legendas) > cabe:                                # passou do teto: entram menos legendas
    legendas = legendas[:max(1, int(len(legendas) * cabe / tokens_de(legendas)))]
por_origem["legendas"] = legendas
todos = [e for o, l in por_origem.items() for _ in range(REPETIR[o]) for e in l]
s_, a_ = empacotar(todos, np.random.default_rng(SEED + 1))
n = len(s_) - len(s_) % CONVERSA_POR_LOTE
SEQ, ALVO = np.stack(s_[:n]), np.stack(a_[:n]); del s_, a_, todos
PASSOS = len(SEQ) // CONVERSA_POR_LOTE
PLANO = {o: {"exemplos": len(l), "vezes": REPETIR[o], "tokens": tokens_de(l) * REPETIR[o]} for o, l in por_origem.items()}
LEGENDAS_INEDITAS = len(legendas) < len(EX.get(("treino", "legendas"), []))
# AGENTS.md: quanto do treino veio do professor (exposições de exemplos de conversa, contando repetições)
_expo = sum(v["exemplos"] * v["vezes"] for v in PLANO.values())
PROFESSOR_FRACAO = round(REL_EXTRAS["v7_casos_por_origem"].get("professor_v7", 0) * REPETIR["v6"] / max(1, _expo), 4)
print(f"fração do treino de conversa vinda do professor: {100 * PROFESSOR_FRACAO:.1f}% das exposições", flush=True)
print("plano:", json.dumps(PLANO, ensure_ascii=False), flush=True)
print(f"PASSOS: {PASSOS} ({len(SEQ)} sequências de conversa + {LOTE - CONVERSA_POR_LOTE} de leitura por passo) | "
      f"{100 * ALVO.mean():.0f}% dos tokens de conversa têm perda", flush=True)
if PASSOS < 50:
    raise SystemExit("ERRO: poucos dados de conversa. Mande o print para o assistente.")

def val_pacotes(parte, origem, max_ex=600):
    lista = EX.get((parte, origem), [])[:max_ex]
    if not lista:
        return None
    s, a = empacotar(lista, np.random.default_rng(SEED))
    return np.stack(s), np.stack(a)
VAL = {f"{p}/{o}": val_pacotes(p, o) for p in ("val", "teste") for o in REPETIR}
VAL = {k: v for k, v in VAL.items() if v is not None}
SELECAO = ["v7c: sentido_dev (todas as frases novas de cena do dev da fase 5 + 600 do dev v7c, estrito, com o IDENTIDADE.md aceito), "
           "com guarda: nenhum grupo do dev (urgência, identidade, conversa, saúde) nem o total abaixo do 46M do chat nas mesmas frases; "
           "concorrem os pesos crus e a média móvel (EMA 0,998 desde 40% do treino) em cada medida; sem versão que passe a guarda, "
           "fica a de maior folga no pior grupo"]
print("medidas separadas:", {k: len(v[0]) for k, v in VAL.items()}, "| escolha da melhor versão por", SELECAO, flush=True)


In [ ]:
# Célula 4 — o CRIVO da 6a, ou o checkpoint
CKPT = SAIDA_R / "checkpoint.pt"
salvo = torch.load(BASE, map_location="cpu", weights_only=True)
CFG = dict(salvo["config"], dropout=0.0)
assert CFG["contexto"] >= CTX, CFG
modelo = lp.LinguagemProfunda(lp.Configuracao(**CFG)).to(dispositivo)
modelo.load_state_dict(salvo["modelo"]); del salvo
PARTIDA_SHA = BASE_SHA
if PARTIDA == "v4":
    modelo.load_state_dict(torch.load(P4, map_location="cpu", weights_only=True)["modelo"])
    PARTIDA_SHA = R4["pesos_sha256"]
    print("partindo do 46M do chat:", P4, flush=True)
print(f"base 6a: {CFG['camadas']} camadas, {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parâmetros, contexto {CFG['contexto']}", flush=True)
PASSO0, HIST, MELHOR_NOTA, MELHOR_ESTADO, MELHOR_PASSO, MELHOR_TIPO, CANDIDATOS, EMA_SALVO = 0, [], None, None, 0, None, [], None
if CKPT.is_file():
    c = torch.load(CKPT, map_location="cpu", weights_only=False)
    if c.get("base_sha256") != PARTIDA_SHA or c.get("passos") != PASSOS:
        raise SystemExit("ERRO: o checkpoint é de outra base ou de outro plano de dados. Apague checkpoint.pt para recomeçar.")
    modelo.load_state_dict(c["modelo"]); PASSO0, HIST, MELHOR_NOTA = c["passo"], c["historico"], c["melhor_nota"]
    MELHOR_ESTADO, MELHOR_PASSO, MELHOR_TIPO = c["melhor_modelo"], c["melhor_passo"], c["melhor_tipo"]
    CANDIDATOS, EMA_SALVO = c["candidatos"], c["ema"]
    print("continuando do checkpoint, passo", PASSO0, flush=True)


In [ ]:
# Célula 5 — treinar: 48 sequências de conversa + 16 de leitura por passo
LR, LR_MIN, AQUECIMENTO = (3e-5, 3e-6, 50) if PARTIDA == "v4" else (6e-5, 6e-6, 100) if PARTIDA == "7a" else (1e-4, 1e-5, 100)
PESO_LEITURA = 0.5
MEDIR_A_CADA = max(50, PASSOS // 9)
MEMORIA_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MICRO = 4 if MEMORIA_GB > 60 else 8 if MEMORIA_GB > 30 else 16   # 290M: A100 80 GB em 4 pedaços; 40 GB em 8; menor em 16
print(f"GPU com {MEMORIA_GB:.0f} GB: lote em {MICRO} pedaço(s)", flush=True)
SALVAR_A_CADA, MAX_HORAS = 1000, 11.5   # 290M: cada checkpoint tem uns 6 GB
EMA_DECAIMENTO, EMA_DESDE = 0.998, int(0.4 * PASSOS)   # média móvel dos pesos (v7c)   # cada checkpoint substituído vai para a lixeira do Drive (e ocupa espaço)

params = list(modelo.parameters())
otim = torch.optim.AdamW([{"params": [p for p in params if p.dim() >= 2], "weight_decay": 0.1},
                          {"params": [p for p in params if p.dim() < 2], "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95), fused=True)
if CKPT.is_file():
    otim.load_state_dict(c["otim"]); del c
EMA = [e.to(dispositivo) for e in EMA_SALVO] if EMA_SALVO else None
escala = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
try:
    rodar = torch.compile(modelo)
except Exception as erro:
    print("sem torch.compile:", erro, flush=True); rodar = modelo
taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR_MIN + (LR - LR_MIN) * 0.5 * (1 + math.cos(math.pi * min(1.0, (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))

def alvos_de(seq, alvo):
    """x = seq[:-1]; y = seq[1:] onde o token tem perda, -100 no resto."""
    s = torch.from_numpy(seq.astype(np.int64))
    y = s[:, 1:].clone(); y[~torch.from_numpy(alvo[:, 1:])] = -100
    return s[:, :-1], y

def leitura(rng, n):
    out = []
    for _ in range(n):
        u = rng.random()
        if u < 0.03:      # o IDENTIDADE.md, inteiro ou um pedaço
            ini = int(rng.integers(0, max(1, len(IDENT) - L)))
            pedaco = IDENT[ini:ini + L]
            s = np.full(L, FIM, np.uint16); s[:len(pedaco)] = pedaco
            a = np.zeros(L, bool); a[1:len(pedaco)] = True
        else:
            fonte = WEB if u < 0.55 else PROPRIO
            i = int(rng.integers(0, len(fonte) - L))
            s = np.asarray(fonte[i:i + L]); a = np.ones(L, bool)
        out.append((s, a))
    return np.stack([s for s, _ in out]), np.stack([a for _, a in out])

def lote(passo):
    i = (passo - 1) * CONVERSA_POR_LOTE
    xc, yc = alvos_de(SEQ[i:i + CONVERSA_POR_LOTE], ALVO[i:i + CONVERSA_POR_LOTE])
    xl, yl = alvos_de(*leitura(np.random.default_rng(SEED * 7 + passo), LOTE - CONVERSA_POR_LOTE))
    return xc.to(dispositivo), yc.to(dispositivo), xl.to(dispositivo), yl.to(dispositivo)

@torch.no_grad()
def perda_pacotes(seq, alvo, tam=None):
    tam = tam or max(2, 16 // MICRO)
    modelo.eval(); soma, n = 0.0, 0
    for j in range(0, len(seq), tam):
        x, y = alvos_de(seq[j:j + tam], alvo[j:j + tam])
        x, y = x.to(dispositivo), y.to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0]
        soma += F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), y.reshape(-1), ignore_index=-100, reduction="sum").item()
        n += int((y != -100).sum())
    return soma / max(1, n)

@torch.no_grad()
def perda_leitura(fonte, ctx, lotes=15, tam=None):
    tam = tam or max(2, 16 // MICRO)
    modelo.eval(); g = np.random.default_rng(SEED); r = []
    for _ in range(lotes):
        ini = g.integers(0, len(fonte) - ctx - 1, size=tam)
        b = torch.from_numpy(np.stack([fonte[i:i + ctx + 1] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            r.append(modelo(b[:, :-1], b[:, 1:])[1].item())
    return float(np.mean(r))

def medir(passo, separadas=("val",)):
    m = {"passo": passo}
    for k, (s, a) in VAL.items():
        if k.split("/")[0] in separadas:
            m["perda_" + k] = perda_pacotes(s, a)
    m["leitura_proprio_256"] = perda_leitura(PROPRIO_VAL, 256)
    m["leitura_web_1024"] = perda_leitura(WEB_VAL, CTX)
    ls_dev = avaliar_blind(DEV_SENTIDO)
    m["sentido_dev"], m["grupos_dev"] = certo_ident(ls_dev), grupos_dev(ls_dev)
    print(f"  MEDIDA passo {passo}: " + " | ".join(f"{k} {v:.4f}" for k, v in m.items() if isinstance(v, float)), flush=True)
    print(f"    grupos do dev: {m['grupos_dev']}" + (f" | chat: {DEV_V4['grupos']}" if DEV_V4 else ""), flush=True)
    return m

def medir_ema(passo):
    """O dev com a média móvel dos pesos no lugar dos crus; depois os crus voltam."""
    crus = [p.detach().clone() for p in params]
    with torch.no_grad():
        for p, e in zip(params, EMA):
            p.copy_(e)
    ls_dev = avaliar_blind(DEV_SENTIDO)
    m = {"passo": passo, "sentido_dev": certo_ident(ls_dev), "grupos_dev": grupos_dev(ls_dev)}
    print(f"  MEDIDA EMA passo {passo}: sentido_dev {m['sentido_dev']:.4f} | grupos do dev: {m['grupos_dev']}", flush=True)
    considerar(m, "ema")   # com a média ainda carregada
    with torch.no_grad():
        for p, c_ in zip(params, crus):
            p.copy_(c_)
    del crus
    return m

def julgar_candidato(m):
    """(passa a guarda, folga no pior grupo em pontos, sentido no dev). Guarda: nenhum grupo do dev nem o total abaixo do chat."""
    if DEV_V4 is None:
        return True, 0.0, m["sentido_dev"]
    folgas = [100 * (m["grupos_dev"][g][0] - DEV_V4["grupos"][g][0]) / max(1, DEV_V4["grupos"][g][1]) for g in GUARDA]
    folgas.append(100 * (m["sentido_dev"] - DEV_V4["sentido_dev"]))
    return min(folgas) >= 0, round(min(folgas), 2), m["sentido_dev"]

def melhor_que(a, b):
    """Quem passa a guarda vence; entre os que passam, mais sentido no dev; entre os que não passam, maior folga no pior grupo."""
    if b is None:
        return True
    if a[0] != b[0]:
        return a[0]
    return (a[2], a[1]) > (b[2], b[1]) if a[0] else (a[1], a[2]) > (b[1], b[2])

def considerar(m, tipo):
    global MELHOR_NOTA, MELHOR_ESTADO, MELHOR_PASSO, MELHOR_TIPO
    n = julgar_candidato(m)
    CANDIDATOS.append({"passo": m["passo"], "tipo": tipo, "sentido_dev": m["sentido_dev"], "grupos_dev": m["grupos_dev"],
                       "passa_guarda": n[0], "folga_pior_grupo": n[1]})
    if melhor_que(n, MELHOR_NOTA):
        MELHOR_NOTA, MELHOR_PASSO, MELHOR_TIPO = n, m["passo"], tipo
        MELHOR_ESTADO = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}   # na memória, não no Drive
        print(f"  ↑ melhor até agora: passo {m['passo']} ({tipo}), sentido no dev {n[2]:.3f}, "
              f"{'passa a guarda' if n[0] else 'NÃO passa a guarda'} (folga no pior grupo {n[1]:+.1f} pontos)", flush=True)

def salvar(passo):
    tmp = SAIDA_R / "checkpoint.tmp"
    torch.save({"modelo": modelo.state_dict(), "otim": otim.state_dict(), "passo": passo, "historico": HIST,
                "melhor_nota": MELHOR_NOTA, "melhor_modelo": MELHOR_ESTADO, "melhor_passo": MELHOR_PASSO, "melhor_tipo": MELHOR_TIPO,
                "candidatos": CANDIDATOS, "ema": [e.detach().cpu() for e in EMA] if EMA is not None else None, "config": CFG, "commit": COMMIT, "base_sha256": PARTIDA_SHA, "passos": PASSOS}, tmp)
    tmp.replace(CKPT)

PROIBIDOS = [ESP["<pad>"], DOC, ESP["<usuario>"], ESP["<assistente>"]]
FREIO = {"penalidade": 1.3, "sem_repetir_ngrama": 3, "top_k": 40}
SEM_FREIO = {"penalidade": 1.0, "sem_repetir_ngrama": 10 ** 9, "top_k": 40}   # como a v4 F foi medida

def caso_ids(c):
    ctx = {"quem": QUEM, **(c["contexto"] or {})}
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(c["historico"])]
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, t + [{"papel": "usuario", "texto": c["mensagem"]}], contexto=CTX - 120, gerar=True)
    return ids

@torch.no_grad()
def gerar_lote_freado(prompts, max_novos=110):
    # Gulosa, várias de uma vez (prompts do mesmo tamanho), com o mesmo freio de repetição da célula 6.
    modelo.eval()
    ids = torch.tensor(prompts, device=dispositivo)
    vistos = torch.zeros(len(prompts), CFG["vocabulario"], dtype=torch.bool, device=dispositivo)
    vivos, saidas, fim = list(range(len(prompts))), [[] for _ in prompts], [False] * len(prompts)
    k = FREIO["sem_repetir_ngrama"] - 1
    for _ in range(max_novos):
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(ids)[0][:, -1].float()
        lg[:, PROIBIDOS] = -float("inf")
        lg = torch.where(vistos, torch.where(lg > 0, lg / FREIO["penalidade"], lg * FREIO["penalidade"]), lg)
        for j, i in enumerate(vivos):
            s_ = saidas[i]
            if len(s_) >= k:
                proib = [s_[a + k] for a in range(len(s_) - k) if s_[a:a + k] == s_[-k:]]
                if proib:
                    lg[j, proib] = -float("inf")
        nxt = lg.argmax(-1).tolist()
        manter = []
        for j, (i, t) in enumerate(zip(vivos, nxt)):
            if t == FIM:
                fim[i] = True
            else:
                saidas[i].append(t); manter.append(j)
        if not manter:
            break
        idx = torch.tensor(manter, device=dispositivo)
        novos = torch.tensor([[nxt[j]] for j in manter], device=dispositivo)
        ids = torch.cat([ids[idx], novos], 1)
        vistos = vistos[idx]; vistos.scatter_(1, novos, True)
        vivos = [vivos[j] for j in manter]
    return saidas, fim

def avaliar_blind(casos, lote_max=None):
    lote_max = lote_max or (96 if MEMORIA_GB > 30 else 32)   # GPU menor: menos frases por vez
    prep = []
    for c in casos:
        try:
            prep.append((c, caso_ids(c)))
        except ValueError:
            pass
    grupos = {}
    for n, (_, p) in enumerate(prep):
        grupos.setdefault(len(p), []).append(n)
    textos, fins = [None] * len(prep), [False] * len(prep)
    for _, idx in sorted(grupos.items()):
        for a in range(0, len(idx), lote_max):
            parte = idx[a:a + lote_max]
            sa, fi = gerar_lote_freado([prep[n][1] for n in parte])
            for n, s_, f_ in zip(parte, sa, fi):
                textos[n], fins[n] = TOK.decode(s_).strip(), f_
    linhas = []
    for (c, _), texto, fim_ in zip(prep, textos, fins):
        av = formato_fase5.avaliar_saida(c, texto, fim_)
        extra = (c.get("exigir") or {}).get("contem_texto")
        if extra:
            av["contem_texto"] = extra.casefold() in texto.casefold()
            av["tudo_ok"] = av["tudo_ok"] and av["contem_texto"]
        linha = {"situacao": c["situacao"], "contraexemplo": bool(c.get("contraexemplo")), "mensagem": c["mensagem"],
                 "alvo": c["resposta"], "saida": texto, **av}
        if c.get("aceita"):
            linha["sentido"] = cenas_v4.julgar_real(sentido, c, texto)
            linha["aceita"], linha["historico"] = c["aceita"], c["historico"][-2:]
        else:
            j = sentido.julgar(linha)
            linha["sentido"] = None if j is None else j[0]
            if j is not None and j[0] != "certo" and sentido.norm(texto) in ACEITAS_IDENT.get(c["situacao"][5:], ()):
                linha["sentido_ident"] = "certo"   # resposta do IDENTIDADE.md nessa situação
            if j is not None and j[0] != "certo" and j[1] & comparar_v6.VIZINHAS.get(c["situacao"][5:], set()):
                linha["sentido_vizinha"] = "certo"   # vizinha da lista fixada antes (conta à parte)
        linha["chave"] = comparar_v6.chave(c)
        linhas.append(linha)
    return linhas

# Grupos da guarda do dev (v7c): situações que caíram no v7 ou no v7b, ou que decidem um critério
GUARDA = {"urgencia": {"saude_urgente", "ajuda_socorro", "ansioso_peito"},
          "identidade": {"quem_e", "nome_crivo", "ser_humano", "robo", "sente", "ser_ia", "confirma_crivo", "consciencia"},
          "conversa": {"reciproco", "estou_bem", "oi_tudo_bem", "tem_certeza", "nao_era_isso", "nao_entendeu_queixa",
                       "voce_confunde", "dificil_falar", "nao_sabe_entao", "quem_sou_eu"},
          "saude": {"saude_geral", "ansioso", "estou_mal", "cansado", "sono", "medo"}}

def grupos_dev(linhas):
    out = {}
    for g, sits in GUARDA.items():
        ls = [l for l in linhas if l["sentido"] is not None and l["situacao"][5:] in sits]
        out[g] = [sum(l["sentido"] == "certo" or l.get("sentido_ident") == "certo" for l in ls), len(ls)]
    return out

def certo_ident(linhas):
    # Sentido certo contando também as respostas do IDENTIDADE.md nas situações que ele cobre.
    cenas = [l for l in linhas if l["sentido"] is not None]
    return round(sum(l["sentido"] == "certo" or l.get("sentido_ident") == "certo" for l in cenas) / max(1, len(cenas)), 4)

def estratificar(casos, n, semente):
    por = {}
    for c in casos:
        por.setdefault(c["situacao"], []).append(c)
    rr = random.Random(semente)
    for v in por.values():
        rr.shuffle(v)
    out = []
    while len(out) < n and any(por.values()):
        for k_ in sorted(por):
            if por[k_] and len(out) < n:
                out.append(por[k_].pop())
    return out

DEV_SENTIDO = ([c for c in F5["v4"]["dev"] if c.get("frase_nova") and c["situacao"].startswith("cena:")]
               + estratificar(V6["dev"], 600, 20261018))   # o mesmo dev nas duas rodadas
print("dev de escolha:", len(DEV_SENTIDO), "frases", flush=True)

# O 46M do chat nas mesmas frases do dev (uma vez; depois vem do Drive): é a régua da guarda
DEV_V4_ARQ = SAIDA / "dev_v4_base.json"
CHAVES_DEV = [comparar_v6.chave(c) for c in DEV_SENTIDO]
DEV_V4 = None
if DEV_V4_ARQ.is_file():
    _d = json.loads(DEV_V4_ARQ.read_text())
    if _d.get("chaves") == CHAVES_DEV and _d.get("pesos_sha256") == R4["pesos_sha256"]:
        DEV_V4 = _d
if DEV_V4 is None and P4 is not None:
    _ls = com_o_chat(lambda: avaliar_blind(DEV_SENTIDO))   # o v7c (46M) nas mesmas frases do dev
    DEV_V4 = {"pesos_sha256": R4["pesos_sha256"], "chaves": CHAVES_DEV, "sentido_dev": certo_ident(_ls), "grupos": grupos_dev(_ls)}
    DEV_V4_ARQ.write_text(json.dumps(DEV_V4, ensure_ascii=False))
    del _ls
if DEV_V4 is None:
    print("AVISO: não achei os pesos do 46M do chat (sha 45825352…): sem guarda, a escolha fica só pelo sentido no dev.", flush=True)
else:
    print(f"guarda: o 46M do chat no dev tem sentido {DEV_V4['sentido_dev']:.3f}; grupos {DEV_V4['grupos']}", flush=True)

if PASSO0 == 0:
    HIST.append(medir(0, ("val", "teste")))   # a partida, antes do ajuste
    if PARTIDA == "v4":   # catraca: o 46M do chat concorre
        considerar(HIST[-1], "chat")
t0, passo = time.time(), PASSO0
for passo in range(PASSO0 + 1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    xc, yc, xl, yl = lote(passo)
    otim.zero_grad(set_to_none=True)
    soma_conv = soma_leit = 0.0
    for parte in range(MICRO):   # GPU menor: o mesmo lote em pedaços, gradiente acumulado
        fc = slice(parte * len(xc) // MICRO, (parte + 1) * len(xc) // MICRO)
        fl = slice(parte * len(xl) // MICRO, (parte + 1) * len(xl) // MICRO)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = rodar(torch.cat([xc[fc], xl[fl]]))[0].float()
        V, nc = lg.shape[-1], fc.stop - fc.start
        l_conv = F.cross_entropy(lg[:nc].reshape(-1, V), yc[fc].reshape(-1), ignore_index=-100)
        l_leit = F.cross_entropy(lg[nc:].reshape(-1, V), yl[fl].reshape(-1), ignore_index=-100) if len(lg) > nc else lg.sum() * 0
        escala.scale((l_conv + PESO_LEITURA * l_leit) / MICRO).backward()
        soma_conv += l_conv.item() / MICRO; soma_leit += l_leit.item() / MICRO
        del lg
    escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    escala.step(otim); escala.update()
    if passo >= EMA_DESDE:   # média móvel dos pesos (v7c)
        with torch.no_grad():
            if EMA is None:
                EMA = [p.detach().float().clone() for p in params]
            else:
                torch._foreach_mul_(EMA, EMA_DECAIMENTO)
                torch._foreach_add_(EMA, [p.detach() for p in params], alpha=1 - EMA_DECAIMENTO)
    if passo % 50 == 0:
        print(f"passo {passo}/{PASSOS} conversa {soma_conv:.3f} leitura {soma_leit:.3f} lr {taxa(passo):.2e} | "
              f"{(time.time()-t0)/max(1,passo-PASSO0):.2f}s/passo", flush=True)
    if passo % MEDIR_A_CADA == 0 or passo == PASSOS:
        HIST.append(medir(passo))
        considerar(HIST[-1], "crus")
        if EMA is not None:
            HIST[-1]["ema"] = medir_ema(passo)
    fim_tempo = time.time() - t0 > MAX_HORAS * 3600
    if passo % SALVAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
        salvar(passo)
        print(f"  checkpoint salvo (passo {passo})", flush=True)
    if fim_tempo:
        print("Limite de tempo da sessão: rode 'Executar tudo' de novo para continuar.", flush=True)
        break
TERMINOU = passo >= PASSOS


In [ ]:
# Célula 6 — a melhor versão: medidas finais e o que ele diz (sem retoque)
if not TERMINOU:
    raise SystemExit("Ainda não terminou: rode 'Executar tudo' de novo para continuar.")
modelo.load_state_dict(MELHOR_ESTADO); PASSO_MELHOR, TIPO_MELHOR = MELHOR_PASSO, MELHOR_TIPO
print(f"escolhida: passo {PASSO_MELHOR} ({TIPO_MELHOR}) | passa a guarda do dev: {MELHOR_NOTA[0]} | folga no pior grupo "
      f"{MELHOR_NOTA[1]:+.1f} pontos | sentido no dev {MELHOR_NOTA[2]:.3f}", flush=True)
if not MELHOR_NOTA[0]:
    print("AVISO: nenhuma versão passou a guarda do dev (ficou abaixo do chat em algum grupo). Ela segue para a régua, "
          "mas o resumo registra.", flush=True)
if PARTIDA == "v4" and PASSO_MELHOR == 0:
    print("AVISO: nenhuma versão treinada passou o 46M do chat no dev; a escolhida é o próprio 46M do chat.", flush=True)
FINAL = medir(PASSO_MELHOR, ("val", "teste"))


@torch.no_grad()
def gerar_freado(ids, max_tokens, temperatura, semente):
    # Como modelo.gerar, com freio de repetição: tokens já ditos perdem força e nenhum trecho de 3 tokens se repete.
    modelo.eval(); g = torch.Generator(device=dispositivo).manual_seed(semente)
    ids, saida = list(ids), []
    for _ in range(max_tokens):
        x = torch.tensor([ids[-CFG["contexto"]:]], dtype=torch.long, device=dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0][0, -1].float()
        lg[PROIBIDOS] = -float("inf")
        if saida:
            vistos = torch.tensor(sorted(set(saida)), device=dispositivo)
            v = lg[vistos]; lg[vistos] = torch.where(v > 0, v / FREIO["penalidade"], v * FREIO["penalidade"])
        k = FREIO["sem_repetir_ngrama"] - 1
        if len(saida) >= k:
            proib = [saida[i + k] for i in range(len(saida) - k) if saida[i:i + k] == saida[-k:]]
            if proib:
                lg[proib] = -float("inf")
        if temperatura == 0:
            prox = int(lg.argmax())
        else:
            lg = lg / temperatura
            lg[lg < lg.topk(FREIO["top_k"]).values[-1]] = -float("inf")
            prox = int(torch.multinomial(lg.softmax(-1), 1, generator=g))
        if prox == FIM:
            return saida, True
        saida.append(prox); ids.append(prox)
    return saida, False

def falar(ctx, turnos, temperatura=0.0, semente=0, max_tokens=120):
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, turnos, contexto=CTX - max_tokens, gerar=True)
    saida, terminou = gerar_freado(ids, max_tokens, temperatura, semente)
    return TOK.decode(saida).strip(), terminou

def turnos_de(historico, mensagem):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(historico)]
    return t + [{"papel": "usuario", "texto": mensagem}]

def conferir(ctx, historico, mensagem, texto, terminou, exigir=None):
    caso = {"contexto": {k: v for k, v in (ctx or {}).items() if k in formato_fase5.CAMPOS}, "historico": list(historico),
            "mensagem": mensagem + " " + " ".join((ctx or {}).get("lembrancas", [])) + " " + (ctx or {}).get("quem", ""),
            "exigir": exigir or {}}
    r = formato_fase5.avaliar_saida(caso, texto, terminou)
    return {k: v for k, v in r.items() if k in ("tudo_ok", "usa_fato", "inventados", "sem_repeticao", "terminou")}

FALAS = []
def ouvir(grupo, ctx, historico, mensagem, exigir=None, espera=None):
    for temp, sem in ((0.0, 0), (0.7, 1)):
        texto, terminou = falar(ctx, turnos_de(historico, mensagem), temp, sem)
        FALAS.append({"grupo": grupo, "temperatura": temp, "historico": list(historico), "mensagem": mensagem,
                      "lembrancas": (ctx or {}).get("lembrancas"), "fatos": (ctx or {}).get("fatos"), "resposta": texto,
                      "terminou": terminou, "conferencia": conferir(ctx, historico, mensagem, texto, terminou, exigir),
                      "lembrou": (espera.casefold() in texto.casefold()) if espera else None})

Q = {"quem": QUEM}
for c in cenas_v4.CONVERSAS_REAIS:
    ouvir("frases_reais_do_criador", Q, c["historico"], c["mensagem"])
for m in ("quem é você?", "qual é o seu nome?", "você é uma pessoa?", "o que você mais gosta de fazer?", "você tem sentimentos?"):
    ouvir("identidade_treinada", Q, [], m)        # formas que estão no treino
for m in extras_fase6b.PERGUNTAS_AVALIACAO + ["do que você é feito?"]:
    ouvir("identidade_nova", Q, [], m)            # formas que não estão no treino
for m in ("me dá uma dica pra estudar melhor", "como faço arroz?", "por que o céu é azul?", "o que é mais importante na vida?",
          "tô cansado do trabalho", "me conta uma curiosidade", "qual a diferença entre vírus e bactéria?", "você gosta de música?"):
    ouvir("perguntas_livres", Q, [], m)
# Valores, tipos e perguntas fora do treino (extras_fase6b.FATOS não tem papagaio, irmã, Juiz de Fora, Remo, filosofia...)
for lembr, msg, espera in ((["a pessoa disse: meu nome é Rosângela"], "você sabe como eu me chamo?", "Rosângela"),
                           (["a pessoa disse: tenho um papagaio chamado Louro"], "qual o nome do meu papagaio?", "Louro"),
                           (["a pessoa disse: eu moro em Juiz de Fora", "a pessoa disse: tenho 40 anos"], "em que cidade eu vivo?", "Juiz de Fora"),
                           (["a pessoa disse: semana que vem tenho prova de filosofia"], "minha prova é de quê?", "filosofia"),
                           (["a pessoa disse: eu torço pro Remo"], "lembra qual é o meu time?", "Remo"),
                           (["a pessoa disse: minha irmã se chama Lúcia", "a pessoa disse: ela tem 7 anos"], "quantos anos tem a minha irmã?", "7"),
                           (["a pessoa disse: meu nome é Tânia"], "qual é o meu time?", "não me contou"),
                           (["a pessoa disse: eu moro em Natal"], "qual é o nome do meu cachorro?", "não me contou")):
    ouvir("lembrancas", {"quem": QUEM, "lembrancas": lembr}, [], msg, espera=espera)
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "acervo" and sum(f["grupo"] == "fatos_do_acervo" for f in FALAS) < 40:
        ouvir("fatos_do_acervo", e["ctx"], [], e["turnos"][0]["texto"], exigir={"fato": True})
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "oasst2" and sum(f["grupo"] == "oasst2_separado" for f in FALAS) < 30             and e["turnos"][-2]["texto"] not in {f["mensagem"] for f in FALAS}:
        ts = e["turnos"][:-1]
        ouvir("oasst2_separado", e["ctx"], [t["texto"] for t in ts[:-1]], ts[-1]["texto"])

def taxa_de(grupo, chave):
    v = [f["conferencia"].get(chave) if chave != "lembrou" else f["lembrou"] for f in FALAS if f["grupo"] == grupo and f["temperatura"] == 0.0]
    v = [x for x in v if x is not None]
    return round(sum(v) / len(v), 3) if v else None
TAXAS = {"conferencia_tudo_ok": {g: taxa_de(g, "tudo_ok") for g in sorted({f["grupo"] for f in FALAS})},
         "usa_fato_acervo": taxa_de("fatos_do_acervo", "usa_fato"), "lembrou": taxa_de("lembrancas", "lembrou")}
for f in FALAS:
    if f["temperatura"] == 0.0 and f["grupo"] in ("frases_reais_do_criador", "identidade_treinada", "identidade_nova", "lembrancas", "perguntas_livres"):
        print(f"[{f['grupo']}] {f['mensagem']!r}\n   → {f['resposta']}", flush=True)
print(json.dumps(TAXAS, ensure_ascii=False, indent=1), flush=True)


In [ ]:
# Célula 6b — a régua da v4 F: sentido nos blinds e nas frases reais, com o contexto que o chat monta
BLINDS = {"v3": F5["v3"]["blind_v3"], "v1": F5["v1"]["blind"], "parafrases": F5["v2"]["blind_parafrase"],
          "grok": F5["v2"]["blind_grok"], "v4": F5["v4"]["blind_v4"], "v6": V6["blind_v6"]}
SENTIDO, LINHAS = {}, {}
for k, casos in BLINDS.items():
    t0 = time.time()
    ls = avaliar_blind(casos)
    a = sentido.analisar(ls, k)
    LINHAS[k] = comparar_v6.compacto(ls)
    SENTIDO[k] = {"certo": a["certo"], "certo_com_identidade": certo_ident(ls), "confundiu": a["confundiu"], "n": a["n_cenas"], "tudo_ok": formato_fase5.resumir(ls).get("tudo_ok"),
                  "piores": a["piores"][:6],
                  "exemplos": [{x: l[x] for x in ("situacao", "mensagem", "saida", "sentido")} for l in ls[::max(1, len(ls) // 12)]][:12]}
    SENTIDO[k]["ic95"] = comparar_v6.ic95([comparar_v6.certo(l) for l in LINHAS[k]])
    SENTIDO[k]["com_vizinhas"] = comparar_v6.ic95([comparar_v6.certo(l, True) for l in LINHAS[k]])["media"]
    f_ = F_SENTIDO.get(k)
    print(f"sentido {k}: {100 * (a['certo'] or 0):.1f}% (lista da F) | {100 * SENTIDO[k]['certo_com_identidade']:.1f}% (com o IDENTIDADE.md)"
          + (f" | v4 F, no chat: {100 * f_:.1f}%" if f_ is not None else "")
          + f" | {time.time() - t0:.0f}s", flush=True)
FREIO_ATUAL = FREIO
FREIO = SEM_FREIO
ls_sem = avaliar_blind(F5["v3"]["blind_v3"])
SEM_FREIO_V3 = {"certo": sentido.analisar(ls_sem, "v3")["certo"], "certo_com_identidade": certo_ident(ls_sem)}
ls_sem = avaliar_blind(F5["v4"]["conversas_reais"])
SEM_FREIO_REAIS = sum(l["sentido"] == "certo" for l in ls_sem)
FREIO = FREIO_ATUAL
print(f"sem freio: sentido v3 {100 * (SEM_FREIO_V3['certo'] or 0):.1f}% (com o IDENTIDADE.md {100 * SEM_FREIO_V3['certo_com_identidade']:.1f}%) | "
      f"frases reais {SEM_FREIO_REAIS}/15", flush=True)
ls = avaliar_blind(F5["v4"]["conversas_reais"])
REAIS = {"certo": sum(l["sentido"] == "certo" for l in ls), "n": len(ls),
         "linhas": [{x: l.get(x) for x in ("historico", "mensagem", "saida", "sentido", "aceita")} for l in ls]}
print(f"\nfrases reais do criador: {REAIS['certo']}/{REAIS['n']} certas (v4 F: 14/15)", flush=True)
for l in REAIS["linhas"]:
    print(f"  [{l['sentido']}] {l['mensagem']!r}\n     → {l['saida']}", flush=True)


# Linha de base pareada: o v7c do chat, nas mesmas frases (uma vez; depois vem do Drive). O nome do arquivo é o que
# comparar_v6.py lê; o conteúdo é o v7c (pesos_sha256 dentro). As linhas da rodada 2 do v7c são o próprio chat: reaproveitadas
# blind a blind quando as frases são as mesmas; o que faltar é medido agora com o 46M.
BASE_LINHAS = SAIDA / "linhas_v4_base.json"
if BASE_LINHAS.is_file() and json.loads(BASE_LINHAS.read_text()).get("pesos_sha256") != CHAT_SHA:
    raise SystemExit(f"ERRO: {BASE_LINHAS} não é do v7c do chat. Apague o arquivo para medir de novo.")
if not BASE_LINHAS.is_file():
    _v7c = CHAT_PASTA / "linhas_rodada2.json"
    _r = json.loads(_v7c.read_text()) if _v7c.is_file() else {"linhas": {}}
    base_linhas, medir_blinds = {}, {}
    for k, casos in BLINDS.items():
        chaves = [comparar_v6.chave(c) for c in casos]
        if [l["chave"] for l in _r["linhas"].get(k, [])] == chaves:
            base_linhas[k] = _r["linhas"][k]
        else:
            medir_blinds[k] = casos
    print("linha de base v7c: reaproveitados", sorted(base_linhas), "| medidos agora", sorted(medir_blinds), flush=True)
    base_linhas.update(com_o_chat(lambda: {k: comparar_v6.compacto(avaliar_blind(c)) for k, c in medir_blinds.items()}))
    reais_chat = _r.get("frases_reais")
    if reais_chat is None or medir_blinds:
        reais_chat = sum(l["sentido"] == "certo" for l in com_o_chat(lambda: avaliar_blind(F5["v4"]["conversas_reais"])))
    BASE_LINHAS.write_text(json.dumps({"pesos_sha256": CHAT_SHA, "modelo": "46M v7c (chat)", "linhas": base_linhas,
                                       "frases_reais": reais_chat}, ensure_ascii=False))
    print("linha de base (v7c do chat) guardada:", BASE_LINHAS, flush=True)
print(comparar_v6.tabela({f"v6 r{RODADA}": LINHAS}), flush=True)
JULGAMENTO = None
if BASE_LINHAS.is_file():
    B = json.loads(BASE_LINHAS.read_text())
    print(comparar_v6.tabela({"v7c (chat)": B["linhas"]}), flush=True)
    JULGAMENTO = comparar_v6.julgar_rodada(LINHAS, B["linhas"], REAIS["certo"], TAXAS["usa_fato_acervo"], com_urgencia=True)
    for k, d in JULGAMENTO["diferencas"].items():
        print(f"  {k:11s} 300M − v7c = {100 * d['media']:+.1f} pontos [{100 * d['ic95'][0]:+.1f}; {100 * d['ic95'][1]:+.1f}]", flush=True)
    print("urgência (acertos 300M / v7c do chat):", JULGAMENTO["urgencia_acertos"], flush=True)
    print("critérios desta rodada:", JULGAMENTO["criterios"], "| passou:", JULGAMENTO["passou"], flush=True)
(SAIDA / f"linhas_rodada{RODADA}.json").write_text(json.dumps({"linhas": LINHAS, "frases_reais": REAIS["certo"],
                                                              "usa_fato": TAXAS["usa_fato_acervo"], "seed": SEED}, ensure_ascii=False))


In [ ]:
# Célula 7 — salvar a versão que conversa e o resumo pequeno
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA_R / f"conversa_{stamp}"
DEST.mkdir(parents=True)
torch.save({"versao": lp.VERSAO, "modelo": {k: v.detach().cpu() for k, v in modelo.state_dict().items()}, "config": CFG,
            "passo": PASSO_MELHOR, "execucao": {"tokenizer_sha256": sha256(BASE.parent / "tokenizer.json"), "pesos_externos": False,
                                                 "base_6a_sha256": BASE_SHA, "protocolo": "fase7b_300m_v7c", "base_7a_sha256": BASE_SHA, "partida": PARTIDA, "partida_sha256": PARTIDA_SHA, "commit": COMMIT, "dados_v6_sha256": M6["dados_sha256"],
                                                 "dados_sha256": MANIFESTO_DADOS["dados_sha256"]}}, DEST / "pesos.pt")
shutil.copyfile(BASE.parent / "tokenizer.json", DEST / "tokenizer.json")
shutil.copyfile(REPO / "experimentos/fase6b_conversa_humana/formato_fase6.py", DEST / "formato_fase6.py")
resumo = {"protocolo": "fase7b_300m_v7c", "regua_contra": {"modelo": "46M v7c (chat)", "pesos_sha256": CHAT_SHA}, "professor_fracao_exposicoes": PROFESSOR_FRACAO, "professor": (MP if USAR_PROFESSOR else None), "partida": PARTIDA, "partida_sha256": PARTIDA_SHA, "lr": [LR, LR_MIN], "escolhida_e_o_chat": PARTIDA == "v4" and PASSO_MELHOR == 0, "substituicao": {"rep_fase5": REP_F5, "rep_v6": REP_V6, "por_situacao": REL_SUBST}, "rodada": RODADA, "seed": SEED, "selecao_v7c": {"escolhida": {"passo": PASSO_MELHOR, "tipo": TIPO_MELHOR, "passa_guarda": MELHOR_NOTA[0], "folga_pior_grupo": MELHOR_NOTA[1], "sentido_dev": MELHOR_NOTA[2]}, "dev_chat": {k: v for k, v in (DEV_V4 or {}).items() if k != "chaves"}, "guarda": {g: sorted(s) for g, s in GUARDA.items()}, "ema": {"decaimento": EMA_DECAIMENTO, "desde_passo": EMA_DESDE}, "candidatos": CANDIDATOS}, "dados_v7c": {k: M6[k] for k in ("dados_sha256", "tamanhos", "falas_por_parte", "falas_por_origem", "casos_por_origem", "origem", "relatorio")}, "julgamento": JULGAMENTO, "criterios_fixados": {**comparar_v6.CRITERIOS, **comparar_v6.CRITERIO_URGENCIA}, "dados_professor": f"{100 * PROFESSOR_FRACAO:.1f}% das exposições de conversa (falas da pessoa; respostas das listas)", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "gpu": GPU,
          "base_6a_sha256": BASE_SHA, "config": CFG, "passos": PASSOS, "passo_escolhido": PASSO_MELHOR, "selecao": SELECAO,
          "plano": PLANO, "extras": REL_EXTRAS, "freio": FREIO, "sentido": SENTIDO, "frases_reais": REAIS, "sem_freio": {"v3": SEM_FREIO_V3, "frases_reais": SEM_FREIO_REAIS}, "f_sentido": F_SENTIDO, "dados_fase5_v4_sha256": V4_SHA, "legendas_acabaram": LEGENDAS_INEDITAS, "dados": MANIFESTO_DADOS,
          "tokens_com_perda": TOKENS_ALVO, "nao_couberam": dict(nao_cabe), "antes": HIST[0], "final": FINAL, "historico": HIST,
          "taxas": TAXAS, "falas": FALAS, "pasta": str(DEST), "pesos_sha256": sha256(DEST / "pesos.pt"),
          "falas_escritas_pelo_assistente": "fase 5 (listas da v4), lembranças (frases curtas), identidade (do IDENTIDADE.md) e as falas da pessoa do v7c (formas_v7*.py, inclusive saúde, pares de saúde e identidade, com autorização do criador); falas da pessoa do professor em dados_professor (modelo de terceiros, não humanas)", "ativo_no_chat": False}
(SAIDA_R / "resumo_v6.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float))
print("SALVO EM", DEST, "| sha", resumo["pesos_sha256"], flush=True)
print(f"Rodada {RODADA} pronta. " + ("Rode 'Executar tudo' de novo para a rodada 2." if RODADA == 1 else "Rode de novo para ver a comparação final.") + " Avise o assistente: ele lê o resumo_v6.json.", flush=True)
